# WILDFIRE PREDICT - FEATURE EXTRACTOR

This module is responsible for loading the downloaded Sentinel-2 images, and running ResNet-18 as feature extractor. The module is created as a Jupyter Notebook to have the option of running the script in `GoogleColab`, if further GPU is required for the processing of the data.

## Libraries

In [1]:
# Libraries
import os
import numpy as np
import torch
import torch.nn as nn


# from google.colab import drive
from scripts.set_parameters import PARAMETERS
from torch.utils.data import Dataset, DataLoader
from torchvision.models import (resnet18, ResNet18_Weights)
from torchvision.models.feature_extraction import create_feature_extractor

## Data Load

### Class `SentinelDataset` 

Create Class to encapsulate and easily manage the Sentinel data

In [2]:
class SentinelData(Dataset):
    def __init__(self, npz_file):
        data      = np.load(npz_file)
        self.x    = data['x']
        self.y    = data['y']
        self.keys = data['composite_key']

    def __len__(self):
        return len(self.x)

    def __getitem__(self, idx):
        # Change/permute image from HeightWidthChannel to CHW as the CNN model expects
        pixel_data = torch.from_numpy(self.x[idx]).permute(2,0,1)
        fire_label = self.y[idx]
        composite_key = str(self.keys[idx])

        return {"pixel_data": pixel_data,
                "fire_label": fire_label,
                "composite_key": composite_key}
    
    def sample_summary(self, idx=0):

        sample = self[idx]

        print("SentinelDataset Sample Summary")
        print("------------------------------")
        print("Showing full data attributes and original attributes vs (->) performed transformations\n")
        print(f"{'Total imgs':<12} : {len(self)}")
        print(f"{'Image shape':<12} : {str(self.x.shape[1:]):<20} -> {tuple(sample['pixel_data'].shape)}")
        print(f"{'Image dtype':<12} : {str(self.x.dtype):<20} -> {sample['pixel_data'].dtype}")
        print(f"{'All labels':<12} : {np.unique(self.y)}")
        print(f"{'Label':<12} : {str(self.y.dtype):<20} -> {sample['fire_label']} ({type(sample['fire_label']).__name__})")
        print(f"{'Key':<12} : {str(self.keys.dtype):<20} -> {sample['composite_key']} ({type(sample['composite_key']).__name__})")

In [3]:
test_file = PARAMETERS['PROJ_HOME']/"data"/"Sentinel2"/"2018_B001_20180101_20180117_sentinel_batch.npz"
test_data = SentinelData(test_file)
test_data.sample_summary()

SentinelDataset Sample Summary
------------------------------
Showing full data attributes and original attributes vs (->) performed transformations

Total imgs   : 787
Image shape  : (128, 128, 3)        -> (3, 128, 128)
Image dtype  : float32              -> torch.float32
All labels   : [False  True]
Label        : bool                 -> False (bool_)
Key          : int64                -> 51620180101 (str)


### Class `ResNetFeatureExtractor`

This class encapsulates the feature extraction process - Turns the Sentinel2 images into 512 feature vector

In [8]:
class ResNetFeatExtractor(nn.Module):
    def __init__(self):
        # Initialize nn.Module class before defining FeatureExtractor
        super().__init__()

        # Load pre trained weights
        weights = ResNet18_Weights.DEFAULT
        self.model = resnet18(weights = weights)
        # Remove final classification layer (as we only need feature extraction)
        self.extractor = create_feature_extractor(self.model, 
                                                  return_nodes = {"avgpool": "features"})
        # Freeze weights
        for param in self.model.parameters():
            param.requires_grad = False

    def forward(self, x):
        features = self.extractor(x)
        return features['features'].flatten(1)

In [ ]:

loader = DataLoader(test_data, batch_size = 32, shuffle = False)

batch = next(iter(loader))
print(batch['pixel_data'].shape) 
print(batch["fire_label"][:5])
print(batch["composite_key"][:5])


In [10]:
model = ResNetFeatExtractor()
img = batch['pixel_data']
with torch.no_grad():
    features = model(img)

print(features.shape)

torch.Size([32, 512])


In [11]:
print(features)

tensor([[0.2250, 1.0879, 0.7368,  ..., 0.2686, 0.6329, 0.0000],
        [0.4735, 0.1369, 2.0451,  ..., 1.0127, 1.3240, 0.2196],
        [0.4397, 1.1269, 1.9944,  ..., 1.3039, 0.1850, 3.5827],
        ...,
        [1.5455, 1.0852, 0.9104,  ..., 0.3212, 1.4653, 0.0407],
        [0.4318, 1.2318, 2.9762,  ..., 0.4681, 0.9135, 0.9103],
        [0.0921, 0.3609, 1.1662,  ..., 0.0000, 0.4455, 0.7643]])


## Processing

In [ ]:
import utils.file_utils as u
import pandas as pd 
# Load files to proces
files = u.get_filepaths(PARAMETERS['DATA_DIR'], "Sentinel2", "npz")
print(f"Total files to process: {len(files)}")
files = files[0:3]
# initialise objects
#rows = []
model = ResNetFeatExtractor()
model.eval()
batch_frames = []

for f in files:
    print(f"Currently at {f}")
    sentinel_data = SentinelData(f)
    loader = DataLoader(sentinel_data, batch_size = 32, shuffle = False)

    for batch in loader:
        images = batch['pixel_data']
        keys   = batch['composite_key']
        
        with torch.no_grad():
            features = model(images).cpu().numpy()
        
        df_batch = pd.DataFrame(features, columns = [f"feat_{i:03d}" for i in range(features.shape[1])])
        df_batch.insert(0, "composite_key", keys)
        batch_frames.append(df_batch)
    break
        
df_features = pd.concat(batch_frames, ignore_index = True)


Total files to process: 49
Currently at /Users/enmanuelmoreno/Documents/Universities/Birkbeck/MSc Data Science/MSc DS Current/MSc Project/MScProjectWildFirePredict/data/Sentinel2/2020_B023_20200730_20200913_sentinel_batch.npz


In [26]:
df_features.head()


,composite_key,feat_000,feat_001,feat_002,feat_003,feat_004,feat_005,feat_006,feat_007,feat_008,...,feat_502,feat_503,feat_504,feat_505,feat_506,feat_507,feat_508,feat_509,feat_510,feat_511
0,55120200730,0.169779,1.136067,1.078956,3.295594,0.995499,1.190337,1.940363,0.596466,0.029648,...,3.675740,0.537765,4.565859,0.742530,0.326533,0.876462,2.464302,0.417797,2.981931,2.596127
1,36920200730,1.738253,0.194225,0.147247,0.646642,0.411605,0.528843,0.541638,0.108117,0.970616,...,0.435601,0.208744,0.221160,0.122932,1.927905,0.139053,1.120345,1.842665,2.665040,1.645739
2,37120200730,4.029680,1.652811,1.263090,1.395288,1.824049,0.009352,1.003603,0.082527,0.655444,...,1.306530,0.259539,2.377685,1.119747,1.266277,4.741567,0.223120,4.009844,1.554152,0.233451
3,111820200730,0.763430,2.036473,0.017336,1.294724,0.248755,0.000000,0.000000,0.000000,0.969376,...,0.497183,0.497147,0.792064,4.691650,0.262358,1.861333,1.611549,3.336388,0.045860,0.014215
4,37520200730,0.624046,1.490577,0.009075,0.109769,0.541859,2.350300,0.909443,0.059520,0.211394,...,0.777877,0.551567,0.000000,1.981180,0.120619,0.026760,0.000000,0.015522,0.904992,1.040734
